In [1]:
# Cell 1 — Install (restart kernel after this cell)
import subprocess
subprocess.run(['pip', 'uninstall', 'unsloth', 'unsloth_zoo', 'torchao', 'trl', 'transformers', 'bitsandbytes', '-y'])
subprocess.run(['pip', 'install', '-q',
    'transformers==4.46.3',
    'peft==0.13.2',
    'accelerate==1.1.1',
    'datasets',
    'seqeval',
    'tqdm',
    'faiss-cpu',
    'sentence-transformers',
])
print('Done. Restart kernel now, then run all cells below.')

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 106.9 MB/s eta 0:00:00
Done. Restart kernel now, then run all cells below.


In [2]:
# Cell 2 — Find adapter path (auto-discovers from notebook output)
import os

ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'adapter_config.json' in files:
        ADAPTER_PATH = root
        break

if ADAPTER_PATH is None:
    raise FileNotFoundError(
        'adapter_config.json not found under /kaggle/input.\n'
        'Add the P-7 training notebook output via: Add data → Your work → p7_train_sricl → Output'
    )
print(f'Adapter found at: {ADAPTER_PATH}')
print('Files:', os.listdir(ADAPTER_PATH))

Adapter found at: /kaggle/input/notebooks/acme105/train-model-for-all-english-datasets-sricl/checkpoints/checkpoint-1022
Files: ['adapter_model.safetensors', 'merges.txt', 'trainer_state.json', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'scheduler.pt', 'special_tokens_map.json', 'optimizer.pt', 'rng_state.pth', 'added_tokens.json']


In [3]:
# Cell 3 — Imports
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

random.seed(42)
torch.manual_seed(42)
print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

2026-05-23 11:27:10.122401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779535630.305180      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779535630.356930      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779535630.818655      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779535630.818691      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779535630.818694      23 computation_placer.cc:177] computation placer alr

Imports OK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Free: 15.5 GB


In [4]:
# Cell 4 — Load all 4 English datasets
say = load_dataset('jjzha/sayfullina')
skl = load_dataset('jjzha/skillspan')
grn = load_dataset('jjzha/green')
gnh = load_dataset('jjzha/gnehm')

for name, ds in [('sayfullina', say), ('skillspan', skl), ('green', grn), ('gnehm', gnh)]:
    print(f'{name:12} train={len(ds["train"]):5}  test={len(ds["test"]):5}')

README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3705 [00:00<?, ? examples/s]

...

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19889 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2557 [00:00<?, ? examples/s]

sayfullina   train= 3705  test= 1851
skillspan    train= 4800  test= 3569
green        train= 8669  test=  335
gnehm        train=19889  test= 2557


In [5]:
# Cell 5 — Tag normalisation + BIO helpers
def normalise_tag(tag):
    if tag in ('B', 'B-SOFT', 'B-ICT', 'B-SKILL',
               'B-PENSEE', 'B-RESULTATS', 'B-RELATIONNEL', 'B-PERSONNEL'):
        return 'B-SKILL'
    if tag in ('I', 'I-SOFT', 'I-ICT', 'I-SKILL',
               'I-PENSEE', 'I-RESULTATS', 'I-RELATIONNEL', 'I-PERSONNEL'):
        return 'I-SKILL'
    return 'O'

def get_tokens_and_tags(example):
    """Merges tags_skill + tags_knowledge (where present) into B-SKILL/I-SKILL/O."""
    tokens = example['tokens']
    merged = ['O'] * len(tokens)
    for col in ('tags_skill', 'tags_knowledge'):
        if col not in example:
            continue
        for i, tag in enumerate(example[col]):
            norm = normalise_tag(str(tag))
            if norm != 'O' and merged[i] == 'O':
                merged[i] = norm
    return tokens, merged

def to_bracket_format(tokens, tags):
    result = []
    i = 0
    while i < len(tokens):
        if tags[i] == 'B-SKILL':
            span = [tokens[i]]
            j = i + 1
            while j < len(tokens) and tags[j] == 'I-SKILL':
                span.append(tokens[j])
                j += 1
            result.append(f"@@{' '.join(span)}##")
            i = j
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)

def bio_from_bracket(tokens, output_text):
    tags = ['O'] * len(tokens)
    text = output_text
    spans = []
    while '@@' in text and '##' in text:
        start = text.find('@@')
        end   = text.find('##', start)
        if start == -1 or end == -1:
            break
        spans.append(text[start+2:end].strip())
        text = text[end+2:]
    for span in spans:
        span_tokens = span.split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == span_tokens:
                if all(tags[i+j] == 'O' for j in range(n)):
                    tags[i] = 'B-SKILL'
                    for j in range(1, n):
                        tags[i+j] = 'I-SKILL'
                break
    return tags

def spans_from_bio(tokens, tags):
    spans, current = [], []
    for token, tag in zip(tokens, tags):
        if tag == 'B-SKILL':
            if current:
                spans.append(' '.join(current))
            current = [token]
        elif tag == 'I-SKILL' and current:
            current.append(token)
        else:
            if current:
                spans.append(' '.join(current))
            current = []
    if current:
        spans.append(' '.join(current))
    return spans

print('Helpers defined')

Helpers defined


In [6]:
# Cell 6 — System prompt (must match training prompt exactly)
SYSTEM_ICL = (
    'You are a skill-span extractor specialising in job advertisement sentences. '
    'Your task is to identify ALL skill spans — both hard skills (tools, technologies, '
    'programming languages, domain knowledge) and soft skills (interpersonal, behavioural, '
    'personal competencies such as communication, leadership, teamwork). '
    'Sentences may be in any language and may be grammatically incomplete. '
    'Extract spans exactly as they appear, token-by-token. '
    'Wrap each skill span with @@...## brackets. '
    'Copy the sentence exactly with no other changes. '
    'If there are no skills, return the sentence unchanged. '
    'Output only the annotated sentence — no explanation, no commentary.'
)

def format_prompt(user_text):
    return (
        f'<|im_start|>system\n{SYSTEM_ICL}<|im_end|>\n'
        f'<|im_start|>user\n{user_text}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

print('Prompt defined')

Prompt defined


In [7]:
# Cell 7 — Build RAG-1 index (full training pool, all 4 datasets)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

ALL_DATASETS = [
    ('sayfullina', say),
    ('skillspan',  skl),
    ('green',      grn),
    ('gnehm',      gnh),
]

all_train_with_skills = []
for name, ds in ALL_DATASETS:
    count = 0
    for ex in ds['train']:
        tokens, tags = get_tokens_and_tags(ex)
        if any(t != 'O' for t in tags):
            all_train_with_skills.append({'tokens': tokens, 'tags': tags, 'source': name})
            count += 1
    print(f'{name}: {count} examples with skills')

print(f'\nTotal RAG-1 pool: {len(all_train_with_skills)}')

rag1_sentences = [' '.join(ex['tokens']) for ex in all_train_with_skills]
print('Embedding RAG-1 sentences...')
rag1_emb = embedder.encode(rag1_sentences, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag1_emb)
rag1_index = faiss.IndexFlatIP(rag1_emb.shape[1])
rag1_index.add(rag1_emb)
print(f'RAG-1 index: {rag1_index.ntotal} entries')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sayfullina: 3701 examples with skills
skillspan: 1584 examples with skills
green: 5136 examples with skills
gnehm: 3408 examples with skills

Total RAG-1 pool: 13829
Embedding RAG-1 sentences...


Batches:   0%|          | 0/217 [00:00<?, ?it/s]

RAG-1 index: 13829 entries


In [8]:
# Cell 8 — Build RAG-2 ESCO index
ESCO_PATH = None
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'skills_en.csv':
            ESCO_PATH = os.path.join(root, f)
            break
    if ESCO_PATH:
        break

if ESCO_PATH is None:
    raise FileNotFoundError('skills_en.csv not found. Add the ESCO dataset via the Data panel.')
print(f'ESCO path: {ESCO_PATH}')

esco = pd.read_csv(ESCO_PATH)

def build_esco_text(row):
    parts = [str(row.get('preferredLabel', ''))]
    alt = str(row.get('altLabels', ''))
    if alt and alt != 'nan':
        parts.append(alt.replace('\n', ', '))
    desc = str(row.get('description', ''))
    if desc and desc != 'nan':
        parts.append(desc[:200])
    return ' | '.join(parts)

esco_texts  = [build_esco_text(row) for _, row in esco.iterrows()]
esco_labels = esco['preferredLabel'].tolist()

print('Embedding ESCO skills...')
rag2_emb = embedder.encode(esco_texts, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag2_emb)
rag2_index = faiss.IndexFlatIP(rag2_emb.shape[1])
rag2_index.add(rag2_emb)
print(f'RAG-2 index: {rag2_index.ntotal} entries')

ESCO path: /kaggle/input/datasets/acme105/skills-en-esco/skills_en.csv
Embedding ESCO skills...


Batches:   0%|          | 0/219 [00:00<?, ?it/s]

RAG-2 index: 13960 entries


In [9]:
# Cell 9 — RAG retrieval + prompt builder
def retrieve_rag1(query_tokens, k=3):
    q_emb = embedder.encode([' '.join(query_tokens)]).astype('float32')
    faiss.normalize_L2(q_emb)
    _, indices = rag1_index.search(q_emb, k)
    return [(all_train_with_skills[i]['tokens'], all_train_with_skills[i]['tags'])
            for i in indices[0]]

def retrieve_rag2(candidate_spans, k=3):
    if not candidate_spans:
        return ''
    definitions = []
    for span in candidate_spans[:5]:
        q_emb = embedder.encode([span]).astype('float32')
        faiss.normalize_L2(q_emb)
        _, indices = rag2_index.search(q_emb, k)
        for idx in indices[0]:
            desc = str(esco.iloc[idx].get('description', ''))
            if desc and desc != 'nan':
                definitions.append(f'- {esco_labels[idx]}: {desc[:120]}')
    seen, unique = set(), []
    for d in definitions:
        if d not in seen:
            seen.add(d)
            unique.append(d)
    return '\n'.join(unique[:8])

def build_user_text(query_tokens, rag1_examples, esco_context):
    lines = []
    if rag1_examples:
        lines.append('Examples of skill annotation:')
        for ex_tokens, ex_tags in rag1_examples:
            lines.append(f'  Input:  {" ".join(ex_tokens)}')
            lines.append(f'  Output: {to_bracket_format(ex_tokens, ex_tags)}')
        lines.append('')
    if esco_context:
        lines.append('Relevant ESCO skill definitions:')
        lines.append(esco_context)
        lines.append('')
    lines.append(' '.join(query_tokens))
    return '\n'.join(lines)

print('RAG functions defined')

RAG functions defined


In [10]:
# Cell 10 — Load base model + saved LoRA adapter
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={'': 0},
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print(f'Model loaded. GPU used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Free GPU: 15.4 GB


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPU used: 6.4 GB


In [11]:
# Cell 11 — BIO Verifier
def verify_bio(tags):
    prev_tag = 'O'
    for i, tag in enumerate(tags):
        if tag == 'O':
            prev_tag = 'O'
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I':
            if prev_tag == 'O':
                return False, f'I-tag at position {i} ({tag}) follows O.'
            _, prev_label = prev_tag.split('-', 1)
            if prev_label != label:
                return False, f'I-tag at position {i} ({tag}) follows {prev_tag} — type mismatch.'
        prev_tag = tag
    return True, ''

def fix_bio(tags):
    fixed = list(tags)
    prev_tag = 'O'
    for i, tag in enumerate(fixed):
        if tag == 'O':
            prev_tag = 'O'
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I':
            if prev_tag == 'O':
                fixed[i] = f'B-{label}'
            else:
                _, prev_label = prev_tag.split('-', 1)
                if prev_label != label:
                    fixed[i] = f'B-{label}'
        prev_tag = fixed[i]
    return fixed

def predict(tokens, max_retries=3):
    rag1_examples = retrieve_rag1(tokens, k=3)
    esco_context  = retrieve_rag2([' '.join(tokens)], k=3)
    prompt        = format_prompt(build_user_text(tokens, rag1_examples, esco_context))
    current_bio   = ['O'] * len(tokens)

    for _ in range(max_retries):
        inputs = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
            )

        decoded = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        )
        current_bio = bio_from_bracket(tokens, decoded)
        valid, error = verify_bio(current_bio)
        if valid:
            return current_bio

        prompt = (
            prompt + decoded + '<|im_end|>\n'
            '<|im_start|>user\n'
            f'BIO error: {error} Output the corrected annotated sentence only.<|im_end|>\n'
            '<|im_start|>assistant\n'
        )

    return fix_bio(current_bio)

print('Verifier + inference defined')

Verifier + inference defined


In [12]:
# Cell 12 — Evaluate all 4 datasets → token-level dataframe
MAX_EXAMPLES = 5000

EVAL_DATASETS = [
    ('Sayfullina', say),
    ('SkillSpan',  skl),
    ('GREEN',      grn),
    ('GNEHM',      gnh),
]

rows = []
example_id = 0

for name, ds in EVAL_DATASETS:
    test_examples = list(ds['test'])
    random.shuffle(test_examples)
    test_examples = test_examples[:MAX_EXAMPLES]

    true_tags, pred_tags = [], []
    for example in tqdm(test_examples, desc=f'Evaluating {name}'):
        tokens, gold = get_tokens_and_tags(example)
        pred = predict(tokens)
        true_tags.append(gold)
        pred_tags.append(pred)

        for token_id, (token, gold_tag, pred_tag) in enumerate(zip(tokens, gold, pred)):
            rows.append({
                'dataset':    name,
                'example_id': example_id,
                'token_id':   token_id,
                'token':      token,
                'gold':       gold_tag,
                'pred':       pred_tag,
            })
        example_id += 1

    f1 = f1_score(true_tags, pred_tags)
    print(f'{name}: F1 = {f1:.4f}')

df = pd.DataFrame(rows)
print(f'\nToken rows: {len(df)}')
df.head(20)

Evaluating Sayfullina:   0%|          | 0/1851 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Evaluating Sayfullina: 100%|██████████

Sayfullina: F1 = 0.4615


Evaluating SkillSpan: 100%|██████████| 3569/3569 [3:49:32<00:00,  3.86s/it]


SkillSpan: F1 = 0.3458


Evaluating GREEN: 100%|██████████| 335/335 [23:13<00:00,  4.16s/it]


GREEN: F1 = 0.2788


Evaluating GNEHM: 100%|██████████| 2557/2557 [4:06:43<00:00,  5.79s/it]


GNEHM: F1 = 0.3812

Token rows: 107517


,dataset,example_id,token_id,token,gold,pred
0,Sayfullina,0,0,work,O,B-SKILL
1,Sayfullina,0,1,in,O,I-SKILL
2,Sayfullina,0,2,different,O,I-SKILL
3,Sayfullina,0,3,way,O,I-SKILL
4,Sayfullina,0,4,enable,O,O
5,Sayfullina,0,5,service,O,O
6,Sayfullina,0,6,user,O,O
7,Sayfullina,0,7,to,O,O
8,Sayfullina,0,8,live,O,O
9,Sayfullina,0,9,more,O,O


In [13]:
# Cell 13 — seqeval F1 summary from dataframe
from seqeval.metrics import classification_report

paper_targets = {
    'Sayfullina': 0.6118,
    'SkillSpan':  0.4665,
    'GREEN':      0.3646,
    'GNEHM':      0.2224,
}

results = {}
for name in df['dataset'].unique():
    sub = df[df['dataset'] == name]
    true_tags = sub.groupby('example_id')['gold'].apply(list).tolist()
    pred_tags = sub.groupby('example_id')['pred'].apply(list).tolist()
    f1 = f1_score(true_tags, pred_tags)
    results[name] = f1
    print(f'\n--- {name} ---')
    print(classification_report(true_tags, pred_tags))

print('=' * 55)
print(f'{"Dataset":<15} {"Our F1":>8} {"Paper F1":>10} {"Delta":>8}')
print('-' * 45)
for name, f1 in results.items():
    paper = paper_targets.get(name, 0)
    print(f'{name:<15} {f1:>8.4f} {paper:>10.4f} {f1 - paper:>+8.4f}')
print('-' * 45)
avg_our   = sum(results.values()) / len(results)
avg_paper = sum(paper_targets.values()) / len(paper_targets)
print(f'{"Average":<15} {avg_our:>8.4f} {avg_paper:>10.4f} {avg_our - avg_paper:>+8.4f}')


--- Sayfullina ---
              precision    recall  f1-score   support

       SKILL       0.46      0.46      0.46      1857

   micro avg       0.46      0.46      0.46      1857
   macro avg       0.46      0.46      0.46      1857
weighted avg       0.46      0.46      0.46      1857


--- SkillSpan ---
              precision    recall  f1-score   support

       SKILL       0.33      0.36      0.35      2247

   micro avg       0.33      0.36      0.35      2247
   macro avg       0.33      0.36      0.35      2247
weighted avg       0.33      0.36      0.35      2247


--- GREEN ---
              precision    recall  f1-score   support

       SKILL       0.47      0.20      0.28       672

   micro avg       0.47      0.20      0.28       672
   macro avg       0.47      0.20      0.28       672
weighted avg       0.47      0.20      0.28       672


--- GNEHM ---
              precision    recall  f1-score   support

       SKILL       0.42      0.35      0.38      1046

  

In [14]:
# Cell 14 — Save token-level dataframe
df.to_csv('/kaggle/working/p7_eval_tokens.csv', index=True)
print('Saved p7_eval_tokens.csv')
print(df.shape)

Saved p7_eval_tokens.csv
(107517, 6)
